In [1]:
import datasets

evaluation_set = datasets.load_from_disk("./kasra_run_evlaution_on_this_original")
evaluation_set

Dataset({
    features: ['REQID_ex', 'completion', 'query', 'class', 'task', 'text', 'label', 'mistral_ai_instruct_7b_chat_hf_preds', 'falcon_7b_base_preds', 'falcon_7b_instruct_preds', 'llama2_7b_chat_hf_preds', 'zephyr_7b_beta_preds', 'compe_gpt_54_2026_03_20', 'claude_sonnet_46_2026_01', 'deepseek_chat_v_32_25_12_01', 'chatgpt4o_frugal_score', 'chatgpt4o_bleu_score', 'chatgpt4o_rougel_score', 'zephyr_frugal_score', 'zephyr_bleu_score', 'zephyr_rougel_score', 'mistralai_frugal_score', 'mistralai_bleu_score', 'mistralai_rougel_score', 'falcon_base_frugal_score', 'falcon_base_bleu_score', 'falcon_base_rougel_score', 'falcon_frugal_score', 'falcon_bleu_score', 'falcon_rougel_score', 'llama_frugal_score', 'llama_bleu_score', 'llama_rougel_score'],
    num_rows: 34
})

In [2]:
query = evaluation_set['query']

instruction = """
Your Role: You are a professional requirements engineer who helps users brainstorm more software requirements in accordance with ISO/IEC/IEEE 29148.

Requirement Writing Rules

Quality Checklist
Every requirement must be:
- Clear: Easily understood without ambiguity.
- Coherent: Logically consistent and well-structured.
- Relevant: Directly addresses the given input.
- Realistic: Feasible within the stated context.
- Implementable: Specifies what is required without prescribing how it should be implemented.

Requirement Type
If the user specifies a requirement type, generate requirements of that type.
If no requirement type is specified, determine the most appropriate type from the request.

A requirement may be functional or non-functional:
- Functional requirements specify the functions, services, or behaviors the system is expected to provide. 
- Non-functional requirements specify quality attributes or constraints on the system. 

Use the following definitions when classifying non-functional requirements:
- Usability: Ease of learning, understanding, or operating the system. 
- Security: Protection against unauthorized access, disclosure, modification, or attack. 
- Operational: Operation in the intended technical or physical environment. 
- Performance: Response time, throughput, or resource use under a stated workload. 
- Look & Feel: Visual appearance, branding, style, or presentation. 
- Scalability: Handling workload changes or growth while maintaining the required service level. 
- Availability: When or how often the system must be operational and accessible. 
- Maintainability: Ease of modifying, correcting, testing, or adapting the software. 
- Legal: Compliance with applicable laws, regulations, licences, or mandated standards. 
- Fault Tolerance: Continued operation despite hardware, software, or component faults.
If the user specifies a non-functional requirement type that is not listed above, generate requirements that align with the requested type according to meaning of the type.

Requirement Format
Follow this template: [Subject] [Signal Word] [Action] [Object] [Constraint]
An Example Requirement Using the Template: The system shall display the user's account dashboard within 2 seconds after a successful login.

- Subject: The actor, such as "The system", "The product", or "The administrator".
- Action: A single active verb, such as "retain", "add", "operate", or "display".
- Object: What the action applies to.
- Constraint: A measurable condition, such as a time, platform, event, or threshold.

Signal Word:
- Shall: For mandatory requirements.
- Should: For preferences or goals.
- Will: For statements of fact or intent.
- May: For optional capabilities or permissions.

Additional Rules
- Base every requirement only on the information provided by the user.
- Do not invent system behavior, constraints, thresholds, actors, or business rules unless they are necessary to produce a valid requirement.
"""

In [3]:
import anthropic

client = anthropic.Anthropic(api_key = "blinded-for-privacy")

In [4]:
def run_claude_model(model_name, query, instruction):
    try:
        response = client.messages.create(
            model = model_name,
            max_tokens = 512,
            temperature = 0.1,
            system = instruction,
            messages =[
                {"role": "user", "content": query}
            ]
        )
        return response.content[0].text
    
    except Exception as e:
        return f"ERROR: {e}"

In [5]:
model_name = 'claude-sonnet-4-6'
claude_sonnet_46_2026_01 = []

for q in query:
    claude_response = run_claude_model(model_name, q, instruction)
    claude_sonnet_46_2026_01.append(claude_response)

In [10]:
evaluation_set = evaluation_set.add_column(f"claude_sonnet_46_2026_01", claude_sonnet_46_2026_01)

In [12]:
evaluation_set.save_to_disk('./models_prediction_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/34 [00:00<?, ? examples/s]